##### Copyright 2024 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## Gemma 2 與 Langfun、PyGlove 和 llama.cpp 的使用指南

[Langfun](https://github.com/google/langfun) 是多功能library，旨在彌合自然語言處理 (NLP) 和結構化資料操作之間的差距。它允許您輕鬆地解析、解釋非結構化文字並將其轉換為定義明確的 Python 物件。對於需要語言模型和資料驅動系統之間無縫整合的應用程式來說，此功能非常寶貴。
[PyGlove](https://github.com/google/pyglove) 是一款強大的 library，可用於靈活且可擴展的組態管理和自動化機器學習 (AutoML)。它提供了用於定義複雜模式和物件表示的工具，使您能夠創建高度可自訂和可維護的程式碼結構。 PyGlove 與 Langfun 的兼容性確保您從自然語言 prompts 產生的物件既強大又易於使用。
[llama.cpp](https://github.com/ggerganov/llama.cpp) 是 Meta AI 的 LLaMA 和其他大型語言模型架構的 C++ 實現，旨在在本地計算機或Google Colab 等環境中實現高效性能。它使您能夠執行大型語言模型，而無需大量計算資源。
[Gemma](https://ai.google.dev/gemma) 是 Google 推出的一系列輕量級、最先進的開源語言模式。 Gemma 模型採用與創建 Gemini 模型相同的研究和技術構建而成，是文本到文本、僅限解碼器的大語言模型 (LLM)，提供英語版本，具有開放權重、預訓練變體和指令調整變體。
Gemma 模型非常適合各種文本生成任務，包括問答、總結和推論。它們相對較小的尺寸使得可以將它們部署在資源有限的環境中，例如筆記型電腦、桌上型電腦或雲端基礎設施，從而實現對最先進人工智慧模型的民主化訪問，並幫助促進每個人的創新。
透過組合 `Langfun`、`PyGlove`、`llama.cpp` 和 `Gemma`，您可以創建不僅能夠理解和處理自然語言，還能將這種理解轉換為結構化、可操作資料的應用程式。這種整合允許您建立複雜的系統，可以使用定義良好的物件與外部APIs、資料庫和其他服務進行交互，從而簡化工作流程並增強功能。
在本指南中，您將學習如何設定環境、設定 Gemma 2 模型以及實現各種實際用例。
<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/.archive/Gemma/[Gemma_2]Using_with_Langfun_and_LlamaCpp.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

## 設定

### 選擇 Colab runtime
要完成本教學，您需要擁有 Colab runtime 以及足夠的資源來執行 Gemma 模型。在這種情況下，您可以使用 T4 GPU：
1. 在 Colab 視窗的右上角，選擇 **▾（其他連接選項）**。
2. 選擇**更改 runtime 類型**。
3. 在 **硬體加速器** 下，選擇 **T4 GPU**。

### Gemma設置

**在深入學習本教學之前，讓我們先設定Gemma：**
1. **Hugging Face 帳戶：** 如果您還沒有帳戶，您可以點選[此處](https://huggingface.co/join) 建立免費的Hugging Face 帳戶。
2. **Gemma 模型存取：** 前往 [Gemma 模型頁面](https://huggingface.co/collections/google/gemma-2-release-667d6600fd5220e7b967f315) 並接受使用條件。
3. **Colab 和 Gemma 電源：** 對於本教學，您需要 Colab runtime 並具有足夠的資源來處理 Gemma 9B 模型. 開始Colab 會話時選擇適當的runtime。
4. **Hugging Face token：** 透過點選[此處](https://huggingface.co/settings/tokens) 產生Hugging Face 存取權限（最好是`write` 權限）token。在本教學的後面部分，您將需要這個token。

**完成這些步驟後，您就可以進入下一部分，在 Colab 環境中設定環境變數。 **

### 設定您的 HF token

將您的 Hugging Face token 新增至 Colab Secrets manager 以安全地儲存它。
1. 開啟 Google Colab notebook 並點選左側面板中的 🔑 Secrets 標籤。 <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
2. 建立一個新的secret，名稱為`HF_TOKEN`。
3. 將token 金鑰複製/貼上到`HF_TOKEN` 的值輸入框中。
4. 切換左側的按鈕以允許notebook 存取secret。


In [ ]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

### 安裝依賴項

You'll need to install a few Python packages for Langfun and PyGlove and some dependencies to interact with HuggingFace along with `llama.cpp` and run the model.

執行以下cell來安裝或升級它：

In [ ]:
# Install ccache, a compiler cache to speed up compilations
!apt install -y ccache

# Install Langfun and PyGlove Python libraries quietly
!pip install -q langfun pyglove

# Upgrade and install the huggingface_hub library for interacting with Hugging Face
!pip install --upgrade -q huggingface_hub

# Clone the llama.cpp repository from GitHub
!rm -rf llama.cpp
!git clone https://github.com/ggerganov/llama.cpp.git

# Navigate into the llama.cpp directory and compile the project with CUDA support
!cd llama.cpp && make -j8 GGML_CUDA=1

您正在編譯 `llama.cpp` 二進位文件，這可能需要一些時間。 Please keep your browser tab active to prevent Colab from shutting down due to inactivity, and return once the compilation is complete to resume working with the notebook.

### 登入Hugging Face Hub

接下來，要進行身份驗證並獲得下載 Gemma 2 模型的存取權限，您需要使用存取權限 token 登入 Hugging Face Hub。

In [ ]:
from huggingface_hub import login

login(os.environ["HF_TOKEN"])

### 下載Gemma 2模型
登入後，您可以從Hugging Face下載Gemma 2個模型檔案。 [Gemma 2 型號](https://huggingface.co/bartowski/gemma-2-9b-it-GGUF) 提供 **GGUF** 格式，該格式針對與`llama.cpp` 一起使用進行了優化。

In [ ]:
from huggingface_hub import hf_hub_download

# Specify the repository and filename
repo_id = 'bartowski/gemma-2-9b-it-GGUF'  # Repository containing the GGUF model
filename = 'gemma-2-9b-it-Q6_K.gguf'        # The GGUF model file

# Download the model file to the current directory
hf_hub_download(repo_id=repo_id, filename=filename, local_dir='.')

### 使用`llama.cpp`執行Gemma 2模型

您將使用 `llama.cpp` library 初始化模型，首先從 HuggingFace 加載預先訓練的 Gemma 2 模型。接下來，您將透過提出一個簡單的問題來 prompt 模型，以便測試 `llama.cpp` 是否可以使用您剛下載的模型。

In [ ]:
!# Define your prompt
prompt = "Q: I have 3 apples and I eat 2. How many apples do I have left?\nA:"
model_path = "gemma-2-9b-it-Q6_K.gguf"

# Run the model using llama.cpp
!./llama.cpp/llama-cli -m "{model_path}" -p "{prompt}" \
  --color --chat-template gemma --temp 0.7 -ngl 9999

接下來，您將使用 `llama-server` 執行伺服器來為 Gemma 2 模型提供服務，從而允許基於 API 的互動。

In [ ]:
!./llama.cpp/llama-server \
  --model "{model_path}" \
  --chat-template gemma \
  --host 0.0.0.0 \
  --port 8000 \
  --chat-template gemma \
  --n-gpu-layers 9999 > server.log 2>&1 &

# Add a delay to allow the server to warm up before making any API requests to
# the server
!sleep 30

提示：如有必要，您可以使用`tail -f server.log` 預覽命令的輸出。

In [ ]:
from google.colab.output import eval_js

# (Optional) View the llama.cpp server interface
# Uncomment the line below to generate a URL for the server
# print(eval_js("google.colab.kernel.proxyPort(8000)"))

注意：取消註解 print 語句將顯示代理 URL，您可以在新的瀏覽器標籤中開啟該 URL 以與伺服器互動。

### 導入庫

為`Langfun` 和`PyGlove` 導入必要的Python 庫，這些庫對於定義模式和與語言模型互動至關重要。這些函式庫提供了有效定義資料模式和查詢語言模型所需的基礎工具。

In [ ]:
import langfun as lf
import pyglove as pg

若要以程式方式將控制 tokens 新增至任何 prompt 與指令調整模型一起使用，您可以建立一個函數，用必要的控制 tokens 包裝 prompt。下面是一個範例 Python 函數，可以重複使用該函數來套用此格式：

In [ ]:
def format_gemma_prompt(user_input):
    """
    Formats a prompt for the Gemma instruction-tuned model by adding control tokens.
    Here you'll be instructing the model to adhere to Langfun and PyGlove.

    Parameters:
    user_input (str): The input from the user to be formatted.

    Returns:
    str: The formatted prompt with Gemma control tokens.
    """
    return f"""\
    You are an AI assistant designed to work seamlessly with Langfun and PyGlove. Your task is to parse user inputs into complex object representations compatible with these libraries. Always output as an object and nothing else. Do not include any additional text, explanations, or comments.

    Guidelines:
    Output Format: Always provide your output as an object representation suitable for Langfun and PyGlove.
    Consistency: Do not deviate from producing the object. Ensure your output is valid and can be used directly in code.
    No Extra Text: Exclude any additional explanations or comments outside of the object.

    Technical Details:
    Your goal is to help parse user inputs into object representations that can be directly utilized within Langfun and PyGlove code.

    <start_of_turn>user
    {user_input}
    <end_of_turn>
    <start_of_turn>model
    """

### 使用 Langfun 與 Google AI 的 Gemma 2

Langfun 可讓您定義清晰的模式並輕鬆地將自然語言輸入轉換為結構化物件。然後，這些物件可以直接插入外部系統、資料庫或外部APIs，從而促進無縫資料流和集成，而無需手動解析或資料清理。
首先，讓我們嘗試一個簡單的用例，請 Langfun 使用 Python 類別 `Answer` 來表示一個簡單數學語句的最終答案。

In [ ]:
class Answer(pg.Object):
  result: int

prompt = 'The result of one plus two is three'
result = lf.query(
    prompt=format_gemma_prompt(prompt),
    schema=Answer,
    lm=lf.llms.LlamaCppRemote("http://0.0.0.0:8000")
)
print(result)

To break down what's happening here.

* `lf.query` 首先將格式化的prompt 傳送到語言模型，並將回應解析為指定的模式。

* `format_gemma_prompt(prompt)` 透過新增必要的控制tokens 和指令來格式化prompt。

* `schema=Answer` 指定輸出應符合`Answer` 模式。

* `lm=lf.llms.LlamaCppRemote("http://0.0.0.0:8000")`使 Langfun 使用執行Gemma 2 9B IT 量化模型的本地`llama.cpp`伺服器。

接下來，您將了解真實世界的用例，並示範如何利用 Langfun 和 Gemma 2 來建立強大的應用程式。

### 將職位描述解析為結構化對象


招募流程自動化可以顯著提高人力資源部門的效率和準確性。透過將職位描述解析為結構化對象，您可以將該數據無縫整合到您的 HR 系統中，從而實現簡歷匹配、職位發布和分析等功能。
首先，我們使用 PyGlove 定義 `Job` 模式，然後提供職位描述作為輸入。 Langfun 由Gemma 2 模型提供支持，處理文字並提取相關信息，相應地填充`Job` 物件。

In [ ]:
# Define the Job schema
class Job(pg.Object):
    title: str
    company: str
    location: str
    responsibilities: list[str]
    requirements: list[str]

# Job description text
job_description = """
We are looking for a Software Engineer to join our team at TechCorp in San Francisco.
Responsibilities include developing software solutions, collaborating with cross-functional teams, and participating in code reviews.
Requirements: Bachelor's degree in Computer Science, 3+ years of experience in software development, proficiency in Python and JavaScript.
"""

# Parse the job description
result = lf.query(
    prompt=format_gemma_prompt(job_description),
    schema=Job,
    lm=lf.llms.LlamaCppRemote("http://0.0.0.0:8000")
)
print(result)

然後，這些結構化資料可以直接在您的 HR 系統或外部 APIs 中用於各種目的，例如職位匹配以及與申請人追蹤系統 (ATS) 集成，以簡化招聘工作流程。

### 啟用支援票證的自動處理

在第二個範例中，您將透過將變數合併到 prompt 中來建立支援票證。在客戶服務應用程式中，有效產生支援票證對於及時解決問題至關重要。透過根據使用者輸入建立結構化支援票證，您可以與票證系統無縫集成，確定問題的優先級，並將其分配給適當的團隊。
定義 `SupportTicket` 模式並在 prompt 中使用 `customer_name` 和 `issue` 等變數。 Langfun 在上下文中處理這些變量，產生結構化的 `SupportTicket` 對象，該對象可以整合到您的客戶服務系統或外部 API 中。這使您可以在系統中自動建立票證、輕鬆追蹤票證狀態並產生報告。

In [ ]:
customer_name = 'John Doe'
issue = 'I am unable to log into my account.'

In [ ]:
# Define the SupportTicket schema
class SupportTicket(pg.Object):
    customer_name: str
    issue: str
    steps_taken: list[str]
    resolution: str | None

# Prompt with variables
prompt = 'Create a support ticket for {{customer_name}} who reported: "{{issue}}" and propose steps to be taken along with a final resolution'

# Generate the support ticket
result = lf.query(
    prompt=format_gemma_prompt(prompt),
    customer_name=customer_name,
    issue=issue,
    schema=SupportTicket,
    lm=lf.llms.LlamaCppRemote("http://0.0.0.0:8000")
)
print(result)

在您的客戶服務團隊每天收到大量詢問的情況下，透過自動建立支援票證，您可以確保一致且有效率地記錄每個問題。结构化的 `SupportTicket` 对象可以直接输入到您的 CRM 或票务软件中，从而加快响​​应时间并提高客户满意度。

### 將客戶回饋解析為結構化數據

了解客戶回饋對於改善產品和服務至關重要。您將首先定義 `Feedback` 架構並提供客戶回饋文字。 Langfun 處理回饋，確定情緒並識別關鍵主題，並輸出結構化的 `Feedback` 物件。這使您可以將客戶回饋解析為結構化數據，有助於情緒分析、報告產生和主題建模。

In [ ]:
# Define the Feedback schema
from typing import Literal

class Feedback(pg.Object):
    customer_id: int
    feedback_text: str
    sentiment: Literal['positive', 'negative', 'mixed']
    topics: list[str]

# Customer feedback text
feedback_text = 'I love the new features in the app, but it crashes frequently when I try to upload photos.'

# Context for the model
context = f'Parse the following customer (id: 1) feedback into an object representation (Feedback): {feedback_text}'

# Parse the feedback
result = lf.query(
    prompt=format_gemma_prompt(context),
    schema=Feedback,
    lm=lf.llms.LlamaCppRemote("http://0.0.0.0:8000")
)
print(result)

此範例展示了模型如何從客戶回饋中提取有價值的見解，這對於產品改進至關重要。想像一下，您管理一個擁有大量用戶群的行動應用程式。透過系統地解析回饋，您可以確定哪些功能受到好評以及哪些領域需要關注。例如，偵測與照片上傳相關的頻繁崩潰可以讓您的開發團隊優先考慮錯誤修復，從而增強整體使用者體驗。

### 使用 `Literal` 類型對支援票證進行分類

對支援票證進行有效分類和優先排序可以顯著增強您的票證分類系統。正確的分類可確保問題由正確的團隊promptly 解決，從而縮短解決時間並提高客戶滿意度。
定義具有特定類別和優先順序的 `Ticket` 模式。提供支援票證描述，Langfun 對其進行處理以對票證進行相應分類。接下來，讓我們使用 `typing` 模組中的 `Literal` 類型執行分類任務，這對於工單分類系統非常有用。

In [ ]:
# Define the Ticket schema
class Ticket(pg.Object):
    id: int
    issue: str
    priority: Literal['low', 'medium', 'high']
    category: Literal['billing', 'technical', 'account', 'other']
    assigned_to: str | None

# Support ticket description
ticket_description = 'Ticket ID: 12345. Customer reports that they are unable to update their billing information on the website.'

# Context for classification
context = f'Classify the following support ticket into the Ticket object: {ticket_description}'

# Classify the ticket
result = lf.query(
    prompt=format_gemma_prompt(context),
    schema=Ticket,
    lm=lf.llms.LlamaCppRemote("http://0.0.0.0:8000")
)
print(result)

在客戶服務中心，票證的性質和緊急程度通常差異很大。透過自動對票證進行分類，您可以確保計費問題由財務團隊處理，技術問題由 IT 部門處理等等。根據故障單的嚴重性確定其優先順序有助於有效管理工作負載並確保快速解決高優先級問題。這展示了該模型自動對支援票證進行分類和優先排序的能力。

###  思考鏈推論

思想鏈 (CoT) 推論涉及將複雜問題分解為一系列邏輯步驟。這種方法透過對每個組件進行系統推論來增強模型解決複雜任務的能力。
在此範例中，您將示範模型對簡單謎題執行 CoT 推論的能力。

In [ ]:
from typing import Optional, List

class Step(pg.Object):
  description: str
  step_output: float

class Solution(pg.Object):
  question: str
  steps: list[Step]
  final_answer: int


# Puzzle question
question = (
  'Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. '
  'She sells the remainder at the farmers\' market daily for $2 per fresh duck egg. '
  'How much in dollars does she make every day at the farmers\' market? '
)

# Assume gemma2_llm is your initialized Gemma 9b model
result = lf.query(
    prompt=format_gemma_prompt(question),
    schema=Solution,
    lm=lf.llms.LlamaCppRemote("http://0.0.0.0:8000")
)
print(result)

這證明了該模型執行思維鏈推論的能力。

在這篇notebook中，您探索如何利用 **Langfun**、**PyGlove**、**llama.cpp** 和 **Hugging Face** 中的 **Gemma 2** 模型來解決各種現實場景。透過實際用例（包括解析工作說明、建立支援票證、分析客戶回饋、對支援票證進行分類以及執行思維鏈推論），您已經了解了 Langfun 如何輕鬆地將非結構化文字轉換為結構化、可操作的物件。這展示了該模型在現實場景中的有效性，增強了應用程式的功能並與外部系統無縫整合。
請隨意擴展這些範例以滿足您的特定需求，並在您的專案中探索 Langfun 和 Gemma 的全部潛力！